# 18.7 Data augmentation strategies

Augmentation teaches invariance by changing examples in ways that should not change their labels. It is synthetic data with a stricter promise: transformations must remain inside the annotation boundary and validation range. Save a copy to Drive to edit.

In [ ]:

import hashlib
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(18)


def dataquality_ladder():
    """Breast Cancer ladder with progressively degraded data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def stable_split(X, y):
    return train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=18,
        stratify=y,
    )


def fit_logistic_accuracy(x_train, y_train, x_test, y_test):
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_train_scaled, y_train)
    pred = clf.predict(x_test_scaled)
    return float(accuracy_score(y_test, pred))


def preview_ladder(rungs):
    for name, X, y in rungs:
        counts = np.bincount(y.astype(int))
        sample = np.round(X[:2, :4], 3)
        print(name)
        print("  shape", X.shape, "class_counts", counts.tolist())
        print("  first_two_by_four")
        print(sample)


## The concept, built once (D1)

For a label-preserving transform $T$, the condition is $y(T(x))=y(x)$. Numeric jitter uses $T(x)=x+\epsilon$; mixup uses $\tilde x=\lambda x_a+(1-\lambda)x_b$ and $\tilde y=\lambda y_a+(1-\lambda)y_b$.

In [ ]:

def lesson_augmentation_checks():
    values = np.array([1.0, 2.0, 3.0])
    jitters = np.array([0.1, -0.1, 0.2])
    augmented = values + jitters
    combined = np.concatenate([values, augmented])
    x_grid = np.linspace(-1.0, 1.0, 80, endpoint=False)
    original_labels = x_grid > 0.0
    shifted_labels = x_grid + 0.8 > 0.0
    disagreements = int(np.sum(original_labels != shifted_labels))
    lam = 0.3
    xa = np.array([0.0, 0.0])
    xb = np.array([2.0, 2.0])
    ya = 0.0
    yb = 1.0
    mix_x = lam * xa + (1.0 - lam) * xb
    mix_y = lam * ya + (1.0 - lam) * yb
    return augmented, values.mean(), combined.mean(), disagreements, disagreements / 80, mix_x, mix_y


augmented, original_mean, combined_mean, disagreements, flip_rate, mix_x, mix_y = lesson_augmentation_checks()
print("augmented", np.round(augmented, 1).tolist())
print("means", original_mean, combined_mean)
print("flip_rate", flip_rate)
print("mixup", mix_x, mix_y)
assert np.allclose(np.round(augmented, 1), [1.1, 1.9, 3.2])
assert round(original_mean, 3) == 2.000
assert round(combined_mean, 3) == 2.033
assert disagreements == 32
assert round(flip_rate, 3) == 0.400
assert np.allclose(mix_x, [1.4, 1.4])
assert round(mix_y, 3) == 0.700


Next, augment standardized training rows with Gaussian jitter and optional within-class mixup. The downstream test set is untouched.

In [ ]:

def augment_train(X, y, radius=0.08, mixup_pairs=80, validate_radius=True):
    x_train, x_test, y_train, y_test = stable_split(X, y)
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    rng = np.random.default_rng(187)
    jitter = rng.normal(0.0, radius, size=x_train_scaled.shape)
    x_jitter = x_train_scaled + jitter
    y_jitter = y_train.copy()
    mix_rows = []
    mix_labels = []

    for cls in np.unique(y_train):
        idx = np.where(y_train == cls)[0]
        if len(idx) < 2:
            continue
        for _ in range(max(1, mixup_pairs // len(np.unique(y_train)))):
            a, b = rng.choice(idx, size=2, replace=False)
            lam = rng.beta(2.0, 2.0)
            row = lam * x_train_scaled[a] + (1.0 - lam) * x_train_scaled[b]
            mix_rows.append(row)
            mix_labels.append(cls)

    if mix_rows:
        x_mix = np.vstack(mix_rows)
        y_mix = np.array(mix_labels)
    else:
        x_mix = np.empty((0, x_train_scaled.shape[1]))
        y_mix = np.array([], dtype=y_train.dtype)

    if validate_radius:
        keep = np.linalg.norm(x_jitter - x_train_scaled, axis=1) <= radius * math.sqrt(x_train_scaled.shape[1]) * 3.0
        x_jitter = x_jitter[keep]
        y_jitter = y_jitter[keep]

    x_aug = np.vstack([x_train_scaled, x_jitter, x_mix])
    y_aug = np.concatenate([y_train, y_jitter, y_mix])
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_aug, y_aug)
    pred = clf.predict(x_test_scaled)
    return {
        "accuracy": float(accuracy_score(y_test, pred)),
        "augmented_rows": int(len(y_aug) - len(y_train)),
        "artifact": x_jitter[:80, :2],
    }


## The dataset ladder

In [ ]:

rungs = dataquality_ladder()
preview_ladder(rungs)


## Run the same method across D1-D5

In [ ]:

results = []
for name, X, y in rungs:
    result = augment_train(X, y)
    row = {
        "rung": name,
        "accuracy": result["accuracy"],
        "augmented_rows": result["augmented_rows"],
        "artifact": result["artifact"],
    }
    results.append(row)
    print(f"{name:28s} acc={row['accuracy']:.3f} augmented={row['augmented_rows']:4d}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row in zip(axes, results):
    art = row["artifact"]
    ax.scatter(art[:, 0], art[:, 1], s=10, alpha=0.7)
    ax.set_title(row["rung"].split()[0])
    ax.set_xlabel("aug f0")
    ax.set_ylabel("aug f1")
fig.suptitle("Label-preserving jitter artifacts")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), [r["accuracy"] for r in results], marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0.6, 1.0)
plt.ylabel("accuracy")
plt.title("Augmented training accuracy by data quality")
plt.grid(True)
plt.show()


## Pitfall on D5: augmenting across the boundary

Too-large transformations become systematic label noise. The fix is a smaller radius plus a validation constraint on augmented displacement.

In [ ]:

name, X, y = rungs[-1]
bad = augment_train(X, y, radius=1.2, mixup_pairs=0, validate_radius=False)
good = augment_train(X, y, radius=0.08, mixup_pairs=80, validate_radius=True)
print("too-large radius accuracy", round(bad["accuracy"], 3))
print("validated small-radius accuracy", round(good["accuracy"], 3))
print("validated augmented rows", good["augmented_rows"])
assert good["augmented_rows"] > 0
assert good["accuracy"] >= bad["accuracy"] - 0.10



## Evaluate it + Practice

- Compare the reported accuracy to a no-skill majority-class baseline for every rung.
- Sanity-check that the key data artifact changes in the intended direction before trusting the metric.
- Ablate the key data-centric step, such as filtering, augmentation, validation, version selection, or valuation-guided dropping.
- Watch for failure signals: gains only on corrupted validation data, impossible feature values, leakage, or unstable row rankings.
- Re-run with a new seed and require the conclusion, not every decimal, to remain stable.

Practice 1: change the corruption or repair strength and re-run the D1 to D5 curve.


Practice 2: add one slice-level diagnostic for the hardest rung.

Practice 3: replace accuracy with balanced accuracy or F1 and compare the story.